# Aula 3 · Lab 3 — Consumidores, grupos e offsets

## Objetivo
O **consumidor** lê mensagens de um tópico. No Kafka, ler **não remove** a mensagem: cada consumidor só
guarda *até onde já leu*. Neste lab vamos entender:

- o que é um **offset** e como o consumidor registra o progresso (**commit**);
- como um **grupo de consumidores** divide as partições entre seus membros para escalar o processamento;
- o **rebalanceamento**, que redistribui as partições quando um membro entra ou sai;
- o **lag**, que mede quanto um grupo está atrasado;
- como **reprocessar** dados voltando o offset (*replay*).

## Conceitos-chave
| Termo | Significado |
|---|---|
| **offset** | posição de uma mensagem dentro de uma partição (0, 1, 2...). Cada partição tem a sua numeração |
| **commit** | o consumidor grava "já processei até o offset X nesta partição". Fica no tópico interno `__consumer_offsets`, separado por **grupo** e por **partição** |
| **grupo de consumidores** (`group.id`) | conjunto de consumidores que dividem o trabalho. Cada partição é lida por **um único** membro do grupo |
| **log-end offset** | offset da próxima mensagem que será escrita na partição (o "fim" atual) |
| **lag** | `log-end offset − offset commitado`: quantas mensagens o grupo ainda não processou |

In [ ]:
import time
from confluent_kafka import Consumer, TopicPartition, OFFSET_BEGINNING
from comum.config import BOOTSTRAP

TOPICO = "aula3-pedidos"

## Etapa 1 — Um consumidor simples

Configurações usadas:

| Parâmetro | Valor | O que faz |
|---|---|---|
| `group.id` | `lab3-notebook` | Nome do grupo. Os commits são guardados em nome do grupo, não do processo: outro consumidor com o mesmo `group.id` continua de onde este parou |
| `auto.offset.reset` | `earliest` | O que fazer quando o grupo **ainda não tem commit** numa partição. `earliest`: começa do início do tópico. `latest`: só lê mensagens que chegarem a partir de agora. Quando já existe commit, essa opção é ignorada |
| `enable.auto.commit` | (padrão `True`) | Faz commit automático dos offsets lidos a cada `auto.commit.interval.ms` (5 s) e também no `close()` |

**O laço de leitura:**
- `subscribe([TOPICO])` entra no grupo. O **coordenador do grupo** (um dos brokers) decide quais partições este consumidor recebe;
- `poll(1.0)` espera até 1 segundo por uma mensagem e devolve `None` se nada chegar;
- `msg.error()` sinaliza eventos especiais, como "fim da partição", que não são mensagens de verdade;
- `close()` faz o commit final e avisa o grupo que este membro saiu.

A célula lê até 15 mensagens ou para depois de 10 segundos.

In [ ]:
c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "lab3-notebook",
              "auto.offset.reset": "earliest"})
c.subscribe([TOPICO])

lidas = 0
fim = time.time() + 10
while time.time() < fim and lidas < 15:
    msg = c.poll(1.0)
    if msg is None or msg.error():
        continue
    lidas += 1
    print(f"p{msg.partition()} off={msg.offset()} chave={msg.key()} valor={msg.value()[:50]}")
c.close()

**Como ler a saída:** `p3 off=12` quer dizer partição 3, offset 12. Observe que:
- os offsets são **crescentes dentro de cada partição**, mas as partições se intercalam: não existe ordem global no tópico;
- as mensagens com a mesma chave (por exemplo, `b'ana'`) sempre vêm da mesma partição, como vimos no Lab 2.

**Rode a célula de novo.** Ela **não** recomeça do zero: continua de onde parou, porque o grupo
`lab3-notebook` já tem offsets commitados. É isso que permite reiniciar um consumidor sem reprocessar tudo.

## Etapa 2 — Grupos e rebalanceamento (no terminal)

Nesta etapa usamos o script `consumidor.py`, que imprime cada mensagem lida e **avisa quando recebe ou
perde partições**. Ele usa a estratégia de atribuição `cooperative-sticky`: num rebalanceamento, cada membro
mantém as partições que já tinha e só as partições "órfãs" mudam de dono. Na estratégia antiga (*eager*),
todos paravam e devolviam tudo a cada mudança no grupo.

### Passo a passo

1. Abra **3 terminais** no Jupyter (*File → New → Terminal*). Em cada um:
   ```bash
   cd ~/notebooks && python aula3_kafka/consumidor.py --grupo analise --inicio latest
   ```
   Cada terminal mostra `+ recebi partições [...]`. Some as listas: as **6 partições** foram divididas
   entre os 3 consumidores, cerca de 2 para cada. Nenhuma partição aparece em dois terminais.

2. Num **4º terminal**, gere tráfego (300 pedidos em 30 s, com 12 chaves diferentes):
   ```bash
   cd ~/notebooks && python -c "
   from confluent_kafka import Producer; import os, time
   p = Producer({'bootstrap.servers': os.environ['KAFKA_BOOTSTRAP']})
   for i in range(300):
       p.produce('aula3-pedidos', key=f'cliente{i%12}', value=f'pedido {i}'); p.poll(0); time.sleep(0.1)
   p.flush()"
   ```
   Cada consumidor só imprime mensagens das **suas** partições, e uma mesma chave aparece sempre no mesmo terminal.

3. **Mate um consumidor** com Ctrl+C. Ele mostra `- perdi partições`, e os outros dois mostram
   `+ recebi partições` com as partições que ficaram órfãs. Isso é o **rebalanceamento**. Como o consumidor saiu
   com `close()`, o grupo é avisado na hora. Se o processo morresse sem aviso, o grupo só perceberia depois de
   `session.timeout.ms` (45 s por padrão) sem sinal de vida (*heartbeat*).

4. **Abra consumidores até ter 7 no grupo.** O 7º não recebe nenhuma partição e fica **ocioso**: com 6 partições,
   no máximo 6 membros trabalham. **O número de partições é o limite de paralelismo de um grupo.**

5. **Abra um consumidor com outro grupo**, por exemplo `--grupo auditoria --inicio earliest`. Ele lê **todas** as
   mensagens do tópico desde o início, sem interferir no grupo `analise`. Cada grupo tem os próprios offsets.

> **Ideia central:** dentro de um grupo, os membros **dividem** o trabalho, e cada partição tem um único dono.
> Entre grupos diferentes, cada um recebe **tudo**. É assim que vários sistemas (analytics, auditoria,
> notificações) consomem o mesmo stream de forma independente.

## Etapa 3 — Lag: quanto o grupo está atrasado?

O **lag** é a métrica número 1 de saúde de um consumidor. No terminal:
```bash
kafka-consumer-groups.sh --bootstrap-server kafka1:9092 --describe --group analise
```

| Coluna | Significado |
|---|---|
| `PARTITION` | partição do tópico |
| `CURRENT-OFFSET` | último offset commitado pelo grupo nessa partição |
| `LOG-END-OFFSET` | offset da próxima mensagem a ser escrita (o "fim" da partição) |
| `LAG` | `LOG-END-OFFSET − CURRENT-OFFSET`: mensagens ainda não processadas |
| `CONSUMER-ID` / `HOST` | qual membro do grupo é dono da partição agora (vazio se o grupo estiver parado) |

**Como interpretar:** lag **estável e baixo** é saudável. Lag **crescendo sem parar** significa que o consumidor
processa mais devagar do que o produtor escreve. As saídas são adicionar membros ao grupo (até o número de
partições), otimizar o processamento ou aumentar as partições.

**Experimente:** rode um consumidor propositalmente lento (0,5 s por mensagem) enquanto o tráfego do passo 2 roda,
e repita o `--describe --group lento` algumas vezes para ver o lag subir:
```bash
python aula3_kafka/consumidor.py --grupo lento --lento 0.5
```

A célula abaixo faz o mesmo cálculo em Python:
- `committed()` devolve o último offset commitado pelo grupo em cada partição;
- `get_watermark_offsets()` devolve o primeiro offset ainda retido e o log-end offset da partição.

In [ ]:
def lag_do_grupo(grupo, topico=TOPICO):
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": grupo, "enable.auto.commit": False})
    parts = [TopicPartition(topico, p) for p in c.list_topics(topico, timeout=10).topics[topico].partitions]
    commitados = c.committed(parts, timeout=10)
    total = 0
    for tp in commitados:
        _, alto = c.get_watermark_offsets(tp, timeout=10)
        atual = tp.offset if tp.offset >= 0 else 0
        print(f"  partição {tp.partition}: commit={atual:<6} fim={alto:<6} lag={alto - atual}")
        total += alto - atual
    c.close()
    print("lag total:", total)

lag_do_grupo("lab3-notebook")

**O que observar:** o grupo `lab3-notebook` leu só 15 mensagens por execução na Etapa 1, então o lag dele é grande:
quase todo o tópico está pendente. Uma partição sem commit aparece com `commit=0`.

## Etapa 4 — Reprocessando dados (replay)

Como o Kafka **retém** as mensagens pelo tempo de `retention.ms`, um consumidor pode voltar no tempo e reler.
Isso é muito usado para:
- corrigir um bug no processamento e **reprocessar** os dados com o código novo;
- popular um sistema novo com o histórico, como um banco ou um índice de busca;
- a **arquitetura Kappa** (Aula 2): em vez de um pipeline batch separado, o "batch" é só reler o stream do início.

São três formas de escolher onde começar a ler.

### 4a. Do início, sem afetar o grupo: `assign()` manual

Até aqui usamos `subscribe()`, em que o **grupo decide** quais partições cada membro lê. Com `assign()`, o
**consumidor escolhe** as partições e o offset inicial, e não participa do rebalanceamento.
`OFFSET_BEGINNING` é um valor especial que significa "o offset mais antigo ainda retido".

Com `enable.auto.commit=False` e sem commit manual, essa leitura não altera nenhum offset salvo: é uma leitura "avulsa".

In [ ]:
c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "lab3-replay", "enable.auto.commit": False})
c.assign([TopicPartition(TOPICO, 0, OFFSET_BEGINNING)])
for _ in range(5):
    m = c.poll(5.0)
    if m and not m.error():
        print(m.partition(), m.offset(), m.value()[:40])
c.close()

**Resultado esperado:** as 5 primeiras mensagens da partição 0, com offsets 0, 1, 2, 3 e 4 (ou a partir do
primeiro offset ainda retido, se a retenção já tiver apagado os mais antigos).

### 4b. A partir de um horário: `offsets_for_times()`

Toda mensagem tem um **timestamp** e o Kafka mantém um índice por tempo em cada partição. Com
`offsets_for_times()` você pergunta: "qual é o primeiro offset com timestamp ≥ T?". Isso é útil para
situações como "o deploy com bug foi às 14h10, reprocesse tudo desde 14h".

Aqui, cada `TopicPartition` recebe o **timestamp em milissegundos** no lugar do offset, e a resposta traz o offset
correspondente. Se nenhuma mensagem da partição for tão recente, a resposta é `-1`, que significa "fim da partição".

In [ ]:
c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "lab3-replay", "enable.auto.commit": False})
cinco_min_atras_ms = int((time.time() - 300) * 1000)
parts = [TopicPartition(TOPICO, p, cinco_min_atras_ms)
         for p in c.list_topics(TOPICO, timeout=10).topics[TOPICO].partitions]
offsets = c.offsets_for_times(parts, timeout=10)
for tp in offsets:
    print(f"partição {tp.partition}: começar no offset {tp.offset}")
c.assign(offsets)
m = c.poll(5.0)
print("primeira mensagem:", m.value()[:60] if m and not m.error() else None)
c.close()

**O que observar:** cada partição tem um offset diferente para o mesmo instante, porque cada partição tem
sua própria numeração. Se tudo for `-1`, nada foi escrito nos últimos 5 minutos: rode o tráfego da Etapa 2 e tente de novo.

### 4c. Resetando os offsets de um grupo inteiro

Para que um grupo **inteiro** passe a ler de outro ponto, alteramos os offsets commitados dele.
O grupo precisa estar **parado**, sem nenhum consumidor ativo: feche os terminais do grupo `analise` antes.

```bash
# mostra o que seria feito, sem alterar nada
kafka-consumer-groups.sh --bootstrap-server kafka1:9092 --group analise --topic aula3-pedidos \
    --reset-offsets --to-earliest --dry-run

# aplica: o grupo volta a ler a partir desse horário (UTC)
kafka-consumer-groups.sh --bootstrap-server kafka1:9092 --group analise --topic aula3-pedidos \
    --reset-offsets --to-datetime 2026-01-01T00:00:00.000 --execute
```

Outras opções de `--reset-offsets`: `--to-latest` (pula tudo), `--shift-by -100` (volta 100 mensagens),
`--to-offset 42` e `--by-duration PT1H` (volta 1 hora).

Depois do reset, abra de novo um `consumidor.py --grupo analise`: ele relê a partir do ponto escolhido.

## Exercícios

Os exercícios 1 e 2 mostram as **semânticas de entrega** da Aula 1, na prática.

1. Desligue o auto-commit (`enable.auto.commit=False`) e faça commit manual (`c.commit(msg, asynchronous=False)`)
   **depois** de processar cada bloco de 10 mensagens. Mate o processo no meio. Ao reiniciar, quantas mensagens são
   relidas? Que semântica é essa: at-least-once ou at-most-once?
2. E se o commit fosse feito **antes** de processar a mensagem? O que poderia acontecer numa falha?
3. Com o gerador da loja rodando num terminal (`python -m comum.gerador --taxa 20`), escreva um consumidor do tópico
   `pedidos` que imprima a **receita acumulada por cidade** a cada 10 segundos.
   Reinicie o consumidor: o total recomeça do zero. Por quê? O offset foi salvo, mas o **estado** (os totais) só existia
   na memória. Salvar estado e offset juntos de forma consistente é o que Spark e Flink fazem com **checkpoint** (Aula 4).